# 库的加载

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import transforms

在我们训练模型的过程中，训练集的形式可能有多种多样，为了支持pytorch的模型训练我们需要为数据集制定一个统一的格式，从而方便用`DataLoader`进行数据加载

在pytorch中有两种格式分别是 `Dataset`和 `IterableDataset`我们可以继承这两个类来，并且重新实现`__init__`,`__len__`,`__getitem__`三个函数来定制属于我们自己的Dataset

* `__init__`:当创建将数据集加载到代码中实例化这个类
* `__len__`:返回这个数据集的大小
* `__getitem__`:返回一个条目及其标签



### `Dataset`类, 他的`__getitem__`能用索引来提取数据

In [10]:
import os
import pandas as pd
from PIL import Image
import numpy as np

class CustomDataset(Dataset):
    def __init__(self,csv_file,image_dir,transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self,idx):
        image_path = os.path.join(self.image_dir,self.annotations.iloc[idx,0])
        print(self.annotations)
        print("loading image from",image_path)
        # 使用 PIL Image 讀取圖像，更穩定且兼容性更好
        image = Image.open(image_path).convert('RGB')
        print("image shape (before transform):", image.size)  # PIL Image 的 size 是 (width, height)
        label = self.annotations.iloc[idx,1]
        print("label",label)
        if self.transform:
            image = self.transform(image)  # ToTensor() 會將 PIL Image 轉換為 [0,1] 範圍的 Tensor (C, H, W)
            print("image shape (after transform):", image.shape)
        return image,label
dataset = CustomDataset(csv_file='/data3/zepeng/LibTutorial/pytorch/dataset/annoation.csv',image_dir='/data3/zepeng/LibTutorial/pytorch/dataset/img',transform=transforms.ToTensor())
image,label = dataset[0]

          img  label
0     cat.png      0
1  dragon.png      1
loading image from /data3/zepeng/LibTutorial/pytorch/dataset/img/cat.png
image shape (before transform): (1408, 768)
label 0
image shape (after transform): torch.Size([3, 768, 1408])


###`IterableData`

之后我们可以直接用dataloader来进行minibatch的数据加载

**返回的`DataLoader`是一个迭代器，每次返回一个batch_size的train_feature和label** 

In [11]:
train_dataloader = DataLoader(dataset=dataset,batch_size=1,shuffle=True)
test_dataloader = DataLoader(dataset=dataset,batch_size=1,shuffle=True)

In [17]:
img,label = next(iter(train_dataloader))
print(img.size)
print(label)
img,label = next(iter(test_dataloader))
print(img.size)
print(label)

          img  label
0     cat.png      0
1  dragon.png      1
loading image from /data3/zepeng/LibTutorial/pytorch/dataset/img/cat.png
image shape (before transform): (1408, 768)
label 0
image shape (after transform): torch.Size([3, 768, 1408])
<built-in method size of Tensor object at 0x7fc8f8b7f110>
tensor([0])
          img  label
0     cat.png      0
1  dragon.png      1
loading image from /data3/zepeng/LibTutorial/pytorch/dataset/img/cat.png
image shape (before transform): (1408, 768)
label 0
image shape (after transform): torch.Size([3, 768, 1408])
<built-in method size of Tensor object at 0x7fc8f8b7e990>
tensor([0])
